#### **Creating Input-Target Pairs**
### This is mainly used to splits the data into input and target data to learn the model and predicts the next token.

In [1]:
import tiktoken

In [2]:
### use the gpt2 tokenizer.
tokenizer = tiktoken.get_encoding("gpt2")



In [4]:
### Get the data from the file

with open("Data/input_target_pairs_data.txt","r") as f:
    raw_text = f.read()

In [5]:
### convert the text into tokens using tokenizer

enc_txt = tokenizer.encode(raw_text)
print(len(enc_txt))

1695


In [11]:
### lets take sample data to test using context size , lets take it has 4 for normal test.enc_txt
context_size = 4

for i in range(1,context_size+1):
    input_id = enc_txt[:i]
    target_id = enc_txt[i]
    print(input_id,"---->",target_id)

[8001] ----> 9542
[8001, 9542] ----> 4430
[8001, 9542, 4430] ----> 318
[8001, 9542, 4430, 318] ----> 5609


In [12]:
### let's see the original tokens
for i in range(1,context_size+1):
    input_id = enc_txt[:i]
    target_id = enc_txt[i]
    context = tokenizer.decode(input_id)
    target = tokenizer.decode([target_id])

    print(context,"---->",target)

Art ----> ificial
Artificial ---->  intelligence
Artificial intelligence ---->  is
Artificial intelligence is ---->  changing


In [14]:
## This is class is used to create a dataset based on input and target data,
#  and return the input data and output data at particular row.
from torch.utils.data import Dataset,DataLoader
import torch
class GPTDataSet(Dataset):
    def __init__(self,txt,tokenizer,max_length,stride):
        self.input_ids = []
        self.target_ids = []

        ## initalize the tokenizer again
        tokenizer = tiktoken.get_encoding("gpt2")

        #conert text into token id's
        token_ids = tokenizer.encode(txt)

        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self,idx):
        return self.input_ids[idx],self.target_ids[idx]

In [15]:
def create_data_loader(txt,batch_size = 4,max_length = 256,stride = 128,
                        shuffle=True,drop_last=True,num_workers = 0):
    
    #tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")
    #dataset
    dataset = GPTDataSet(txt,tokenizer,max_length,stride)

    dataloader = DataLoader(dataset,batch_size=batch_size,shuffle=shuffle,drop_last=drop_last,num_workers=num_workers)


    return dataloader

In [17]:
dataloader = create_data_loader(raw_text,batch_size=4,max_length=16,stride = 8, shuffle=False, num_workers=0)


data_tier = iter(dataloader)

first_batch = next(data_tier)

input,target = first_batch

print("Input:",input)
print("Target:",target)

Input: tensor([[ 8001,  9542,  4430,   318,  5609,   262,   835,   661,  9427,   351,
          3037,    13,   220,   198, 31439,  3788],
        [ 9427,   351,  3037,    13,   220,   198, 31439,  3788,  3341,   460,
          1833,  3303,    11,  7564,  7572,    11],
        [ 3341,   460,  1833,  3303,    11,  7564,  7572,    11, 16602,  1321,
            11,   220,   198,   392,  7716,  4465],
        [16602,  1321,    11,   220,   198,   392,  7716,  4465,  9109,    13,
          2312,  9889,   389,  3170,  1262, 16113]])
Target: tensor([[ 9542,  4430,   318,  5609,   262,   835,   661,  9427,   351,  3037,
            13,   220,   198, 31439,  3788,  3341],
        [  351,  3037,    13,   220,   198, 31439,  3788,  3341,   460,  1833,
          3303,    11,  7564,  7572,    11, 16602],
        [  460,  1833,  3303,    11,  7564,  7572,    11, 16602,  1321,    11,
           220,   198,   392,  7716,  4465,  9109],
        [ 1321,    11,   220,   198,   392,  7716,  4465,  9109,   

In [20]:
#lets  check the shape
print("Input Shape:",input.shape)
print("Target Shape:",target.shape)

Input Shape: torch.Size([4, 16])
Target Shape: torch.Size([4, 16])


In [22]:
print(input[0])

tensor([ 8001,  9542,  4430,   318,  5609,   262,   835,   661,  9427,   351,
         3037,    13,   220,   198, 31439,  3788])


In [27]:
print("Input:",tokenizer.decode(input[1].tolist()))
print("Target:",tokenizer.decode(target[1].tolist()))

Input:  interact with technology. 
Modern software systems can understand language, recognize patterns,
Target:  with technology. 
Modern software systems can understand language, recognize patterns, analyze
